# Chengjie 

In [7]:
from datetime import datetime,timedelta
from matplotlib import pyplot as plt
import numpy as np
import os
from datetime import datetime,timezone

In [8]:
from file_finder import find_files_in_range
from side_config import find_calib_files_for_datafile
from sql import get_values_in_timerange
from mossbauer import mossbauer

In [9]:
from L0 import L0Driver
from raw import epix 

In [10]:
import mysql.connector
class sql_writer:
	def __init__(self,
				 host='192.168.2.2',
				 user='writer',
				 password='mossbauer_writer',
				 database='slowcontrol',
				 table='science_run1'):
		self.table = table  
		self.conn = mysql.connector.connect(
			host=host, user=user, password=password, database=database,
			autocommit=True, connection_timeout=5
		)
		self.cur = self.conn.cursor()
		

In [11]:
# We fetch the data from slowcontrol and write the results into the daq;
import mysql.connector
class sql_writer2:
	def __init__(self,
				 host='192.168.2.2',
				 user='writer',
				 password='mossbauer_writer',
				 database='daq',
				 table='daq'):
		self.table = table  
		self.conn = mysql.connector.connect(
			host=host, user=user, password=password, database=database,
			autocommit=True, connection_timeout=5
		)
		self.cur = self.conn.cursor()
		

In [12]:
sql= sql_writer()
sql2 = sql_writer()

In [13]:
tend=datetime.now()- timedelta(hours=1)
tstart=tend-timedelta(hours=1)

files=find_files_in_range('/data/share',tstart,tend)

In [14]:
data_selector= np.zeros((44,192))
start=3
end=45
up=3
down=42
data_selector[up:down,start:end]=1
data_selector[up:down,start+48:end+48]=1
data_selector[up:down,48*2+start:48*2+end]=1
data_selector[up:down,48*3+start:48*3+end]=1

In [ ]:
get_values_in_timerange(sql,tstart,tend,value_col='block_number')

In [18]:
ds=[]
for i in range(4):
    dss= np.zeros((44,192))
    dss[up:down,start+48*i:end+48*i]=1
    ds.append(dss) 

In [19]:
from tqdm import tqdm
fc=np.empty((1,4))
bc=np.empty((1,4))


for file in tqdm(files):
    det=mossbauer(file)
    dat=det.load_data()
    nframes=np.shape(dat)[0]
    
    t_counts=np.zeros((nframes,4))
    for i in range(nframes):
        for j in range(4):
            t_counts[i,j]=np.sum(dat[i]*ds[j])
    
    findex= np.where( det.get_word(6)==1)[0]
    bindex= np.where( det.get_word(6)==0)[0]

    fc=np.concatenate((fc,t_counts[findex]),axis=0)
    bc=np.concatenate((bc,t_counts[bindex]),axis=0)

    #print(file, np.max(t_counts[findex]))

fc= fc[1:]
bc= bc[1:]

100%|██████████| 1/1 [00:01<00:00,  1.05s/it]


In [21]:
det.get_all_datetime()

array(['2026-04-03T07:35:34.394172', '2026-04-03T07:35:34.621919',
       '2026-04-03T07:35:35.394131', ..., '2026-04-03T08:35:32.703443',
       '2026-04-03T08:35:33.495983', '2026-04-03T08:35:33.703538'],
      shape=(7200,), dtype='datetime64[us]')

In [36]:
np.mean(fc,axis=0)

array([59126.55722222, 56865.65222222, 62904.87527778, 61141.43055556])

In [38]:
det.read_time_range()

(datetime.datetime(2026, 4, 2, 11, 35, 33, 323870),
 datetime.datetime(2026, 4, 2, 12, 35, 32, 650698))

In [17]:
TIME
Q0_B
Q1_B
Q2_B
Q3_B
Q0_F
Q1_F
Q2_F
Q3_F
DataRate

memmap([209025687, 209025786, 209026087, ..., 210464986, 210465287,
        210465386], shape=(7200,), dtype=uint32)

In [16]:
det.load_head()[:,8]

memmap([209026085, 209026184, 209026485, ..., 210465384, 210465685,
        210465784], shape=(7200,), dtype=uint32)

In [20]:
((det.load_head()[:,8] - det.load_head()[:,4])/2 +1 )[det.load_head()[:,6]] 



array([200., 200., 200., ..., 200., 200., 200.], shape=(7200,))

In [28]:
det.load_head()[:,5]

memmap([200, 200, 200, ..., 200, 200, 200], shape=(7200,), dtype=uint32)